In [1]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import copy

# ── 1. Pull data ──────────────────────────────────────────────────────────────
client = bigquery.Client(project="cbb6790-final-project")
query = """
SELECT *
FROM `cbb6790-final-project.analysis.CBB5790_FinalProject`
"""
df = client.query(query).to_dataframe()

# ── 2. Create binary target ───────────────────────────────────────────────────
df["long_stay"] = (df["icu_length_of_stay"] >= 7).astype(int)
df["gender_bin"] = (df["gender"] == "M").astype(int)

FEATURES = [
    "anchor_age",
    "creatinine_min", "creatinine_max",
    "bun_min", "bun_max",
    "potassium_min", "potassium_max",
    "bicarbonate_min", "bicarbonate_max",
    "sodium_min", "sodium_max",
    "mbp_min", "mbp_mean", "mbp_max",
    "heart_rate_min", "heart_rate_max",
    "urineoutput_24hr",
    "kdigo_stage",
    "gender_bin",
]
TARGET     = "long_stay"
N_FEATURES = len(FEATURES)
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── 3. Stratified 80/10/10 split within each care unit ───────────────────────
train_dfs, tune_dfs, test_dfs = [], [], []
skipped_units = []

for unit in df["first_careunit"].dropna().unique():
    unit_df = df[df["first_careunit"] == unit].copy()
    if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
        skipped_units.append((unit, len(unit_df), "too small or no outcome variation"))
        continue
    try:
        unit_trainval, unit_test = train_test_split(
            unit_df, test_size=0.10, random_state=42, stratify=unit_df[TARGET]
        )
        unit_train, unit_tune = train_test_split(
            unit_trainval, test_size=0.1111, random_state=42, stratify=unit_trainval[TARGET]
        )
        train_dfs.append(unit_train)
        tune_dfs.append(unit_tune)
        test_dfs.append(unit_test)
    except ValueError as e:
        skipped_units.append((unit, len(unit_df), str(e)))

df_train = pd.concat(train_dfs).reset_index(drop=True)
df_tune  = pd.concat(tune_dfs).reset_index(drop=True)
df_test  = pd.concat(test_dfs).reset_index(drop=True)

print("── Split sizes ──")
print(f"  Train: {len(df_train)} | Tune: {len(df_tune)} | Test: {len(df_test)}")

print("\n── Rows per care unit across splits ──")
for unit in df["first_careunit"].dropna().unique():
    n_train = (df_train["first_careunit"] == unit).sum()
    n_tune  = (df_tune["first_careunit"]  == unit).sum()
    n_test  = (df_test["first_careunit"]  == unit).sum()
    print(f"  {unit}: train={n_train}, tune={n_tune}, test={n_test}")

if skipped_units:
    print("\n── Skipped units ──")
    for unit, n, reason in skipped_units:
        print(f"  {unit} (n={n}): {reason}")

# ── 4. Global preprocessor (fit on train only) ────────────────────────────────
global_imputer = SimpleImputer(strategy="median")
global_scaler  = StandardScaler()
X_train_imp    = global_imputer.fit_transform(df_train[FEATURES])
global_scaler.fit(X_train_imp)

def preprocess(split_df):
    X = global_imputer.transform(split_df[FEATURES])
    X = global_scaler.transform(X)
    y = split_df[TARGET].values
    return (
        torch.tensor(X, dtype=torch.float32).to(DEVICE),
        torch.tensor(y, dtype=torch.float32).to(DEVICE),
    )

# ── 5. Model definition (LayerNorm instead of BatchNorm1d) ───────────────────
class ICUNet(nn.Module):
    def __init__(self, n_features, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden_dims:
            # LayerNorm works on any batch size, including 1
            layers += [nn.Linear(in_dim, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)

# ── 6. Local training function ────────────────────────────────────────────────
def train_local_dl(local_df, features, target, hidden_dims, dropout,
                   lr, epochs, batch_size, global_weights=None):
    imputer = SimpleImputer(strategy="median")
    scaler  = StandardScaler()
    X = imputer.fit_transform(local_df[features])
    X = scaler.fit_transform(X)
    y = local_df[target].values

    X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    y_t = torch.tensor(y, dtype=torch.float32).to(DEVICE)

    dataset    = TensorDataset(X_t, y_t)
    # drop_last=True discards any final batch of size 1
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    model = ICUNet(len(features), hidden_dims, dropout).to(DEVICE)

    if global_weights is not None:
        model.load_state_dict(copy.deepcopy(global_weights))

    pos_weight = torch.tensor([(y == 0).sum() / max((y == 1).sum(), 1)],
                               dtype=torch.float32).to(DEVICE)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    model.train()
    for epoch in range(epochs):
        for X_batch, y_batch in dataloader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        scheduler.step()

    return {
        "weights":   copy.deepcopy(model.state_dict()),
        "n_samples": len(local_df),
        "imputer":   imputer,
        "scaler":    scaler,
        "unit":      None,
    }

# ── 7. Federated aggregation (FedAvg on model weights) ───────────────────────
def federated_aggregate_dl(client_results):
    total_n    = sum(r["n_samples"] for r in client_results)
    global_wts = copy.deepcopy(client_results[0]["weights"])

    for key in global_wts:
        global_wts[key] = torch.zeros_like(global_wts[key], dtype=torch.float32)
        for r in client_results:
            weight = r["n_samples"] / total_n
            global_wts[key] += weight * r["weights"][key].float()

    return global_wts

# ── 8. Scoring helper ─────────────────────────────────────────────────────────
def score_global_dl(global_weights, hidden_dims, dropout, eval_df, features, target):
    model = ICUNet(len(features), hidden_dims, dropout).to(DEVICE)
    model.load_state_dict(global_weights)
    model.eval()

    X_t, y_t = preprocess(eval_df)
    with torch.no_grad():
        logits = model(X_t)
        probs  = torch.sigmoid(logits).cpu().numpy()

    return roc_auc_score(eval_df[target].values, probs)

# ── 9. Tune hyperparameters on validation set ─────────────────────────────────
param_grid = [
    {"hidden_dims": [64, 32],       "dropout": 0.3, "lr": 1e-3, "epochs": 30, "batch_size": 32},
    {"hidden_dims": [128, 64],      "dropout": 0.3, "lr": 1e-3, "epochs": 30, "batch_size": 32},
    {"hidden_dims": [128, 64, 32],  "dropout": 0.4, "lr": 5e-4, "epochs": 50, "batch_size": 64},
    {"hidden_dims": [64, 32],       "dropout": 0.2, "lr": 5e-4, "epochs": 50, "batch_size": 32},
    {"hidden_dims": [256, 128, 64], "dropout": 0.4, "lr": 1e-3, "epochs": 30, "batch_size": 64},
]
units = df_train["first_careunit"].dropna().unique()

print("\n── Tuning on validation set ──")
tune_results = {}

for params in param_grid:
    client_results = []
    for unit in units:
        unit_df = df_train[df_train["first_careunit"] == unit].copy()
        if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
            continue
        result = train_local_dl(unit_df, FEATURES, TARGET, **params)
        result["unit"] = unit
        client_results.append(result)

    if not client_results:
        continue

    global_weights = federated_aggregate_dl(client_results)
    tune_auc = score_global_dl(
        global_weights, params["hidden_dims"], params["dropout"],
        df_tune, FEATURES, TARGET
    )
    key = str(params)
    tune_results[key] = (tune_auc, params)
    print(f"  {params} → Tune AUC: {tune_auc:.4f}")

best_key    = max(tune_results, key=lambda k: tune_results[k][0])
best_params = tune_results[best_key][1]
print(f"\nBest params: {best_params}")
print(f"Tune AUC:    {tune_results[best_key][0]:.4f}")

# ── 10. Multi-round federated training with best params ───────────────────────
N_ROUNDS       = 3
global_weights = None

print(f"\n── Federated training: {N_ROUNDS} rounds ──")
for round_num in range(N_ROUNDS):
    print(f"\n  Round {round_num + 1}")
    client_results = []

    for unit in units:
        unit_df = df_train[df_train["first_careunit"] == unit].copy()
        if len(unit_df) < 30 or unit_df[TARGET].nunique() < 2:
            continue
        result = train_local_dl(
            unit_df, FEATURES, TARGET,
            global_weights=global_weights,
            **best_params
        )
        result["unit"] = unit
        client_results.append(result)

    global_weights = federated_aggregate_dl(client_results)

    round_auc = score_global_dl(
        global_weights, best_params["hidden_dims"], best_params["dropout"],
        df_tune, FEATURES, TARGET
    )
    print(f"    Clients trained: {len(client_results)}")
    print(f"    Tune AUC after round {round_num + 1}: {round_auc:.4f}")

# ── 11. AUC Evaluation ────────────────────────────────────────────────────────
print("\n── Global Model AUC ──")
for split_name, split_df in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
    auc = score_global_dl(
        global_weights, best_params["hidden_dims"], best_params["dropout"],
        split_df, FEATURES, TARGET
    )
    print(f"  {split_name}: {auc:.4f}")

print("\n── Per-Unit Local Model Test AUC ──")
for r in client_results:
    try:
        model = ICUNet(N_FEATURES, best_params["hidden_dims"], best_params["dropout"]).to(DEVICE)
        model.load_state_dict(r["weights"])
        model.eval()

        X = r["imputer"].transform(df_test[FEATURES])
        X = r["scaler"].transform(X)
        X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)

        with torch.no_grad():
            probs = torch.sigmoid(model(X_t)).cpu().numpy()

        auc = roc_auc_score(df_test[TARGET], probs)
        print(f"  {r['unit']}: AUC = {auc:.4f}")
    except ValueError:
        print(f"  {r['unit']}: AUC could not be computed")

Using device: cpu
── Split sizes ──
  Train: 24456 | Tune: 3061 | Test: 3061

── Rows per care unit across splits ──
  Medical Intensive Care Unit (MICU): train=7614, tune=952, test=952
  Surgical Intensive Care Unit (SICU): train=2509, tune=314, test=314
  Medical/Surgical Intensive Care Unit (MICU/SICU): train=5136, tune=642, test=642
  Trauma SICU (TSICU): train=1864, tune=234, test=234
  Coronary Care Unit (CCU): train=3661, tune=458, test=458
  Cardiac Vascular Intensive Care Unit (CVICU): train=2740, tune=343, test=343
  Neuro Surgical Intensive Care Unit (Neuro SICU): train=310, tune=39, test=39
  Neuro Intermediate: train=416, tune=52, test=52
  Intensive Care Unit (ICU): train=0, tune=0, test=0
  PACU: train=38, tune=5, test=5
  Neuro Stepdown: train=77, tune=10, test=10
  Surgery/Vascular/Intermediate: train=91, tune=12, test=12
  Medicine: train=0, tune=0, test=0
  Surgery/Trauma: train=0, tune=0, test=0
  Medicine/Cardiology Intermediate: train=0, tune=0, test=0

── Skipped

/tmp/ipykernel_3946/341563461.py:152: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  {'hidden_dims': [128, 64, 32], 'dropout': 0.4, 'lr': 0.0005, 'epochs': 50, 'batch_size': 64} → Tune AUC: 0.7908
  {'hidden_dims': [64, 32], 'dropout': 0.2, 'lr': 0.0005, 'epochs': 50, 'batch_size': 32} → Tune AUC: 0.7687


/tmp/ipykernel_3946/341563461.py:152: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


  {'hidden_dims': [256, 128, 64], 'dropout': 0.4, 'lr': 0.001, 'epochs': 30, 'batch_size': 64} → Tune AUC: 0.7943

Best params: {'hidden_dims': [256, 128, 64], 'dropout': 0.4, 'lr': 0.001, 'epochs': 30, 'batch_size': 64}
Tune AUC:    0.7943

── Federated training: 3 rounds ──

  Round 1


...

    Clients trained: 11
    Tune AUC after round 1: 0.7943

  Round 2


/tmp/ipykernel_3946/341563461.py:152: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


    Clients trained: 11
    Tune AUC after round 2: 0.8031

  Round 3


/tmp/ipykernel_3946/341563461.py:152: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


    Clients trained: 11
    Tune AUC after round 3: 0.8022

── Global Model AUC ──
  Train: 0.8300
  Tune: 0.8022
  Test: 0.8137

── Per-Unit Local Model Test AUC ──
  Medical Intensive Care Unit (MICU): AUC = 0.8078
  Surgical Intensive Care Unit (SICU): AUC = 0.7880
  Medical/Surgical Intensive Care Unit (MICU/SICU): AUC = 0.7956
  Trauma SICU (TSICU): AUC = 0.7849
  Coronary Care Unit (CCU): AUC = 0.7742
  Cardiac Vascular Intensive Care Unit (CVICU): AUC = 0.7961
  Neuro Surgical Intensive Care Unit (Neuro SICU): AUC = 0.7726
  Neuro Intermediate: AUC = 0.7756
  PACU: AUC = 0.8102
  Neuro Stepdown: AUC = 0.7940
  Surgery/Vascular/Intermediate: AUC = 0.7870


In [2]:
# ── 12. Centralized Baseline (Pooled Data) ────────────────────────────────────
print("\n── Centralized Baseline Training ──")

X_train_t, y_train_t = preprocess(df_train)

central_dataset = TensorDataset(X_train_t, y_train_t)
central_dataloader = DataLoader(
    central_dataset,
    batch_size=best_params["batch_size"],
    shuffle=True,
    drop_last=True
)

central_model = ICUNet(
    N_FEATURES,
    best_params["hidden_dims"],
    best_params["dropout"]
).to(DEVICE)

y_train_np = y_train_t.cpu().numpy()
pos_weight = torch.tensor(
    [(y_train_np == 0).sum() / max((y_train_np == 1).sum(), 1)],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(
    central_model.parameters(),
    lr=best_params["lr"],
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

central_model.train()
for epoch in range(best_params["epochs"]):
    for X_batch, y_batch in central_dataloader:
        optimizer.zero_grad()
        loss = criterion(central_model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
    scheduler.step()

print("\n── Centralized Model AUC ──")
central_model.eval()
with torch.no_grad():
    for split_name, split_df in [("Train", df_train), ("Tune", df_tune), ("Test", df_test)]:
        X_eval_t, y_eval_t = preprocess(split_df)
        logits = central_model(X_eval_t)
        probs = torch.sigmoid(logits).cpu().numpy()
        auc = roc_auc_score(split_df[TARGET].values, probs)
        print(f"  {split_name}: {auc:.4f}")


── Centralized Baseline Training ──

── Centralized Model AUC ──
  Train: 0.8425
  Tune: 0.8113
  Test: 0.8240
